<a href="https://colab.research.google.com/github/rafaelpimentacruz-stack/python_sis_26_2/blob/main/Jorge_Cruz_Aula11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

#Atividade sala 1

#Atividade01
#Etapa1

# =====================================================================
# ETAPA 1: Importações e Padrão PEP 8
# =====================================================================
import os
import xml.etree.ElementTree as ET

import openpyxl
from openpyxl.styles import Font


# =====================================================================
# PREPARAÇÃO: Criando pasta e 3 amostras simuladas de NF-e (XML)
# =====================================================================
pasta_xml = "notas_fiscais"
os.makedirs(pasta_xml, exist_ok=True)

amostras_nfe = {
    "nfe_001.xml": """<?xml version="1.0" encoding="UTF-8"?>
<nfeProc>
    <NFe>
        <infNFe>
            <ide>
                <nNF>1001</nNF>
                <dhEmi>2026-11-20T10:15:00-03:00</dhEmi>
            </ide>
            <emit>
                <CNPJ>12345678000199</CNPJ>
                <xNome>Tech Suprimentos LTDA</xNome>
            </emit>
            <total>
                <ICMSTot>
                    <vNF>4500.00</vNF>
                </ICMSTot>
            </total>
        </infNFe>
    </NFe>
</nfeProc>""",
    "nfe_002.xml": """<?xml version="1.0" encoding="UTF-8"?>
<nfeProc>
    <NFe>
        <infNFe>
            <ide>
                <nNF>1002</nNF>
                <dhEmi>2026-11-22T14:30:00-03:00</dhEmi>
            </ide>
            <emit>
                <CNPJ>98765432000155</CNPJ>
                <xNome>Servidores Cloud Brasil SA</xNome>
            </emit>
            <total>
                <ICMSTot>
                    <vNF>12890.50</vNF>
                </ICMSTot>
            </total>
        </infNFe>
    </NFe>
</nfeProc>""",
    "nfe_003.xml": """<?xml version="1.0" encoding="UTF-8"?>
<nfeProc>
    <NFe>
        <infNFe>
            <ide>
                <nNF>1003</nNF>
                <dhEmi>2026-11-25T09:45:00-03:00</dhEmi>
            </ide>
            <emit>
                <CNPJ>45612378000110</CNPJ>
                <xNome>Redes e Cabos Express</xNome>
            </emit>
            <total>
                <ICMSTot>
                    <vNF>1250.75</vNF>
                </ICMSTot>
            </total>
        </infNFe>
    </NFe>
</nfeProc>""",
}

for nome_arquivo, conteudo_xml in amostras_nfe.items():
    caminho_completo = os.path.join(pasta_xml, nome_arquivo)
    with open(caminho_completo, "w", encoding="utf-8") as f:
        f.write(conteudo_xml)


# =====================================================================
# ETAPAS 2 e 3: Coleta, Varredura e Extração Segura de Dados Fiscais
# =====================================================================
def extrair_tag(raiz, caminho_tag, padrao="Não informado"):
    """Busca uma tag no XML e retorna seu conteúdo com tratamento de erro."""
    try:
        elemento = raiz.find(caminho_tag)
        if elemento is not None and elemento.text is not None:
            return elemento.text.strip()
        return padrao
    except AttributeError:
        return padrao


dados_extraidos = []

for arquivo in sorted(os.listdir(pasta_xml)):
    if arquivo.lower().endswith(".xml"):
        caminho_xml = os.path.join(pasta_xml, arquivo)

        try:
            arvore = ET.parse(caminho_xml)
            raiz = arvore.getroot()

            numero_nota = extrair_tag(raiz, ".//nNF")
            data_emissao = extrair_tag(raiz, ".//dhEmi")[:10]
            nome_fornecedor = extrair_tag(raiz, ".//emit/xNome")
            cnpj_emitente = extrair_tag(raiz, ".//emit/CNPJ")
            valor_bruto = extrair_tag(raiz, ".//ICMSTot/vNF", padrao="0.00")

            try:
                valor_total = float(valor_bruto)
            except ValueError:
                valor_total = 0.0

            dados_extraidos.append(
                {
                    "numero_nota": numero_nota,
                    "data_emissao": data_emissao,
                    "fornecedor": nome_fornecedor,
                    "cnpj": cnpj_emitente,
                    "valor_total": valor_total,
                }
            )

        except ET.ParseError:
            print(f"Erro: O arquivo '{arquivo}' está mal formatado ou corrompido.")
        except Exception as erro:
            print(f"Erro inesperado ao processar '{arquivo}': {erro}")


# =====================================================================
# ETAPA 4: Geração da Planilha Excel (.xlsx) e Formatação Monetária
# =====================================================================
wb = openpyxl.Workbook()
ws = wb.active
ws.title = "Auditoria NF-e"

cabecalho = [
    "Número da Nota",
    "Data de Emissão",
    "Fornecedor / Emitente",
    "CNPJ do Emitente",
    "Valor Total (R$)",
]
ws.append(cabecalho)

# Estiliza o cabeçalho em negrito
for celula in ws[1]:
    celula.font = Font(bold=True)

# Insere as linhas extraídas e formata o valor monetário
for nota in dados_extraidos:
    # Aplica a formatação com duas casas decimais (:.2f)
    valor_formatado = f"{nota['valor_total']:.2f}"

    ws.append(
        [
            nota["numero_nota"],
            nota["data_emissao"],
            nota["fornecedor"],
            nota["cnpj"],
            valor_formatado,
        ]
    )

    # Também aplica máscara numérica/monetária na célula do Excel
    celula_valor = ws.cell(row=ws.max_row, column=5)
    celula_valor.number_format = "R$ #,##0.00"

    print(
        f"NF {nota['numero_nota']} | {nota['fornecedor']} | "
        f"Valor: R$ {valor_formatado}"
    )

arquivo_saida = "auditoria_notas_fiscais.xlsx"
wb.save(arquivo_saida)
print(f"\nConsolidação concluída! Planilha '{arquivo_saida}' salva com sucesso.")

NF 1001 | Tech Suprimentos LTDA | Valor: R$ 4500.00
NF 1002 | Servidores Cloud Brasil SA | Valor: R$ 12890.50
NF 1003 | Redes e Cabos Express | Valor: R$ 1250.75

Consolidação concluída! Planilha 'auditoria_notas_fiscais.xlsx' salva com sucesso.
